# Real-Time Mode
## One line turns a micro-batch stream into sub-second streaming.

*Running on open-source **Apache Spark 4.1** + **Unity Catalog OSS** - no proprietary runtime.*

In [ ]:
# Run me first - starts Spark and a live stream of order events (~10s).
import sys, time, os
sys.path.insert(0, '/home/jovyan/demos/realtime-mode')
import rtm_stream_lib as rtm
from pyspark.sql import SparkSession
from IPython.display import Code
spark = SparkSession.builder.config('spark.sql.streaming.realTimeMode.allowlistCheck','false').getOrCreate()
spark.sparkContext.setLogLevel('OFF')  # keep the booth screen clean
rtm.start_producer(rows_per_sec=20)
print('Streaming order events...')

## Orders stream through guardrails
Every order event is checked - oversized total, too many items, a leaked credential - and routed **ALLOW** or **QUARANTINE**. The checks are stateless, which is exactly what Real-Time Mode needs:

In [ ]:
Code(filename='/home/jovyan/demos/realtime-mode/rtm_stream_lib.py')

## Turning on Real-Time Mode is one line
It isn't a new query - it's the **same stream with a different trigger**:

```python
# Normal micro-batch:
writer.trigger(processingTime='5 seconds').start()

# Real-Time Mode - same query, only the trigger (note: 5 MINUTES, yet still sub-second latency):
rt = spark._jvm.org.apache.spark.sql.streaming.Trigger.RealTime('5 minutes')
writer._jwrite.trigger(rt).start()
```

### First, the normal micro-batch trigger

In [ ]:
from IPython.display import display
q = rtm.start_query(spark, use_realtime=False, query_name='mb')
time.sleep(12)   # micro-batch: a batch every 5 seconds
display(spark.sql('SELECT decision, count(*) AS n FROM mb GROUP BY decision').toPandas())
display(spark.sql("SELECT order_id, brand, total, reasons FROM mb WHERE decision='QUARANTINE' LIMIT 5").toPandas())
q.stop()

### Now Real-Time Mode - same query, only the trigger changed

In [ ]:
q = rtm.start_query(spark, use_realtime=True, query_name='rt')   # the only change: use_realtime=True
time.sleep(15)   # Real-Time Mode: 5-minute interval, but records arrive in real time
display(spark.sql('SELECT decision, count(*) AS n FROM rt GROUP BY decision').toPandas())
display(spark.sql("SELECT order_id, brand, total, reasons FROM rt WHERE decision='QUARANTINE' LIMIT 5").toPandas())
q.stop()

### What just happened
The guardrail query never changed - only the trigger did. `Trigger.RealTime` is Spark 4.1's Real-Time Mode, here on open-source Spark. It needs a Kafka source and stateless operations, both true above.